In [1]:
# Cell 1: Initialize Streamlit dashboard development

from pathlib import Path
import sys

PROJECT_DIR = Path(r"C:\Hehe\AI-Lead-Desk-Runtime-Errors")
SRC_DIR = PROJECT_DIR / "src"
PACKAGE_DIR = SRC_DIR / "ai_lead_desk"

required_files = [
    PACKAGE_DIR / "rag_service.py",
    PACKAGE_DIR / "voice_service.py",
    PROJECT_DIR / ".env",
]

print("AI LEAD DESK — STREAMLIT DASHBOARD")
print("=" * 45)
print("Project:", PROJECT_DIR)
print("Notebook: 05_streamlit_dashboard.ipynb")

print("\nChecking existing project files:")

all_available = True

for file_path in required_files:
    exists = file_path.exists()
    print(f"[{'OK' if exists else 'MISSING'}] {file_path.relative_to(PROJECT_DIR)}")

    if not exists:
        all_available = False

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

print("\nStreamlit dashboard preparation:",
      "READY" if all_available else "CHECK MISSING FILES")

print("No API requests were made.")

AI LEAD DESK — STREAMLIT DASHBOARD
Project: C:\Hehe\AI-Lead-Desk-Runtime-Errors
Notebook: 05_streamlit_dashboard.ipynb

Checking existing project files:
[OK] src\ai_lead_desk\rag_service.py
[OK] src\ai_lead_desk\voice_service.py
[OK] .env

Streamlit dashboard preparation: READY
No API requests were made.


In [2]:
# Cell 2: Create the Streamlit AI Lead Desk application

from pathlib import Path

APP_SOURCE = r"""
import os
import sys
import tempfile
from pathlib import Path

import streamlit as st
from dotenv import load_dotenv
from google import genai

PROJECT_DIR = Path(__file__).resolve().parent
SRC_DIR = PROJECT_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

load_dotenv(PROJECT_DIR / ".env")

from ai_lead_desk.rag_service import DocumentRAGService
from ai_lead_desk.voice_service import process_voice_message


# ---------------------------------------------------------
# Page configuration
# ---------------------------------------------------------

st.set_page_config(
    page_title="AI Lead Desk",
    page_icon="🎓",
    layout="wide",
)

st.title("AI Lead Desk")
st.caption(
    "AI Admissions Counselor | Document-grounded answers | Voice support"
)


# ---------------------------------------------------------
# Initialize existing AI services
# ---------------------------------------------------------

@st.cache_resource(show_spinner="Loading institute knowledge base...")
def get_rag_service(project_dir):
    return DocumentRAGService(project_dir=project_dir)


@st.cache_resource
def get_gemini_client():
    load_dotenv(PROJECT_DIR / ".env")
    api_key = os.getenv("GEMINI_API_KEY")

    if not api_key:
        return None

    return genai.Client(api_key=api_key)


try:
    rag_service = get_rag_service(str(PROJECT_DIR))
except Exception as exc:
    st.error("Could not initialize the RAG service.")
    st.code(f"{type(exc).__name__}: {exc}")
    st.info(
        "Check the .env file, Groq API key, Ollama service, "
        "and local embedding model."
    )
    st.stop()

try:
    gemini_client = get_gemini_client()
except Exception as exc:
    gemini_client = None
    st.sidebar.warning(
        f"Voice service initialization failed: {type(exc).__name__}"
    )


# ---------------------------------------------------------
# Session state
# ---------------------------------------------------------

if "messages" not in st.session_state:
    st.session_state.messages = [
        {
            "role": "assistant",
            "content": (
                "Hello! Ask me about courses, fees, duration, schedules, "
                "syllabus, and policies covered by institute documents."
            ),
            "answer_mode": "welcome",
            "grounded": None,
            "sources": [],
        }
    ]

if "followups" not in st.session_state:
    st.session_state.followups = []

if "last_voice_result" not in st.session_state:
    st.session_state.last_voice_result = None


# ---------------------------------------------------------
# Sidebar
# ---------------------------------------------------------

documents_dir = PROJECT_DIR / "data" / "institute_documents"
documents_dir.mkdir(parents=True, exist_ok=True)

supported_extensions = {".pdf", ".docx", ".txt", ".md"}

document_files = sorted(
    path for path in documents_dir.iterdir()
    if path.is_file() and path.suffix.lower() in supported_extensions
)

st.sidebar.title("Counselor Workspace")

st.sidebar.metric(
    "Indexed chunks",
    len(rag_service.document_chunks),
)

st.sidebar.metric("Institute documents", len(document_files))
st.sidebar.metric("Counselor follow-ups", len(st.session_state.followups))

language_label = st.sidebar.selectbox(
    "Answer language",
    ["Auto-detect", "English", "Hindi", "Marathi"],
)

language_map = {
    "Auto-detect": None,
    "English": "en",
    "Hindi": "hi",
    "Marathi": "mr",
}

response_language = language_map[language_label]

st.sidebar.divider()
st.sidebar.success("RAG service initialized")

if gemini_client:
    st.sidebar.success("Voice service configured")
else:
    st.sidebar.warning("Voice unavailable: check GEMINI_API_KEY")

st.sidebar.caption(
    "Development prototype. WhatsApp and Instagram integrations "
    "are not connected."
)


# ---------------------------------------------------------
# Evidence display
# ---------------------------------------------------------

def show_sources(record):
    with st.expander("Evidence and sources"):
        if record.get("grounded") is True:
            st.write(
                "The RAG service marked this answer as grounded. "
                "This does not guarantee that every generated claim "
                "has been independently verified."
            )
        else:
            st.warning("This answer was not marked as grounded.")

        st.write("Answer mode:", record.get("answer_mode", "unknown"))

        sources = record.get("sources") or []

        if not sources:
            st.write("No retrieved sources were returned.")
            return

        for source in sources:
            st.write("Document:", source.get("source", "Unknown"))
            st.write("Chunk:", source.get("chunk_id", "N/A"))
            st.write("Similarity:", source.get("similarity", "N/A"))
            st.divider()


def queue_followup(question, answer, sources, answer_mode):
    already_exists = any(
        item["question"] == question
        for item in st.session_state.followups
    )

    if not already_exists:
        st.session_state.followups.append({
            "question": question,
            "answer": answer,
            "sources": sources or [],
            "answer_mode": answer_mode,
        })


# ---------------------------------------------------------
# Main dashboard tabs
# ---------------------------------------------------------

text_tab, voice_tab, knowledge_tab, followup_tab = st.tabs([
    "Text Inquiries",
    "Voice Inquiries",
    "Knowledge Base",
    "Counselor Follow-ups",
])


# ---------------------------------------------------------
# Text inquiries
# ---------------------------------------------------------

with text_tab:
    st.subheader("Student conversation")

    for index, message in enumerate(st.session_state.messages):
        with st.chat_message(message["role"]):
            st.markdown(message["content"])

            if (
                message["role"] == "assistant"
                and message.get("answer_mode") != "welcome"
            ):
                show_sources(message)

                if message.get("question"):
                    if st.button(
                        "Flag for counselor follow-up",
                        key=f"followup_{index}",
                    ):
                        queue_followup(
                            message["question"],
                            message["content"],
                            message.get("sources", []),
                            message.get("answer_mode", "unknown"),
                        )
                        st.rerun()

    question = st.chat_input("Enter a student's question...")

    if question:
        st.session_state.messages.append({
            "role": "user",
            "content": question,
        })

        try:
            with st.spinner("Searching institute documents..."):
                result = rag_service.answer(
                    question,
                    response_language=response_language,
                )

            reply = {
                "role": "assistant",
                "content": str(result["answer"]),
                "question": question,
                "grounded": result.get("grounded", False),
                "sources": result.get("sources", []),
                "answer_mode": result.get("answer_mode", "unknown"),
            }

        except Exception as exc:
            st.error(f"Question processing failed: {type(exc).__name__}: {exc}")

            reply = {
                "role": "assistant",
                "content": (
                    "I couldn't process this inquiry. "
                    "Please refer it to an admissions counselor."
                ),
                "question": question,
                "grounded": False,
                "sources": [],
                "answer_mode": "processing_error",
            }

        st.session_state.messages.append(reply)
        st.rerun()


# ---------------------------------------------------------
# Voice inquiries
# ---------------------------------------------------------

with voice_tab:
    st.subheader("Student voice message")

    st.write(
        "Upload a recording to transcribe the question, retrieve "
        "institute information, and generate a spoken reply."
    )

    audio_file = st.file_uploader(
        "Choose an audio recording",
        type=["wav", "mp3", "m4a", "ogg", "webm", "mp4"],
        key="voice_upload",
    )

    if audio_file is not None:
        st.audio(audio_file)

    if gemini_client is None:
        st.warning("Configure GEMINI_API_KEY to enable voice processing.")

    process_disabled = (
        audio_file is None
        or gemini_client is None
    )

    if st.button(
        "Process voice message",
        type="primary",
        disabled=process_disabled,
    ):
        temporary_path = None

        try:
            if audio_file.size > 25 * 1024 * 1024:
                st.error("Audio file exceeds the 25 MB limit.")
            else:
                output_dir = PROJECT_DIR / "data" / "voice_outputs"
                output_dir.mkdir(parents=True, exist_ok=True)

                suffix = Path(audio_file.name).suffix.lower() or ".wav"

                with tempfile.NamedTemporaryFile(
                    suffix=suffix,
                    delete=False,
                ) as temporary:
                    temporary.write(audio_file.getvalue())
                    temporary_path = Path(temporary.name)

                with st.spinner(
                    "Transcribing, answering, and generating speech..."
                ):
                    result = process_voice_message(
                        audio_path=temporary_path,
                        rag_service=rag_service,
                        gemini_client=gemini_client,
                        output_dir=output_dir,
                    )

                st.session_state.last_voice_result = result
                st.success("Voice message processed.")

        except Exception as exc:
            st.error(f"Voice processing failed: {type(exc).__name__}: {exc}")

        finally:
            if temporary_path and temporary_path.exists():
                temporary_path.unlink()

    voice_result = st.session_state.last_voice_result

    if voice_result:
        st.divider()

        st.markdown("**Transcript**")
        st.write(voice_result.get("transcript", ""))

        st.markdown("**Detected language**")
        st.write(voice_result.get("language", "unknown"))

        st.markdown("**Counselor answer**")
        st.markdown(voice_result.get("answer", ""))

        show_sources(voice_result)

        audio_path = voice_result.get("audio_path")

        if audio_path and Path(audio_path).is_file():
            st.markdown("**Generated voice reply**")
            st.audio(Path(audio_path).read_bytes(), format="audio/wav")

        with st.expander("Processing timings"):
            st.json(voice_result.get("timings", {}))

        if st.button("Flag voice inquiry for counselor follow-up"):
            queue_followup(
                voice_result.get("transcript", ""),
                voice_result.get("answer", ""),
                voice_result.get("sources", []),
                voice_result.get("answer_mode", "unknown"),
            )
            st.rerun()


# ---------------------------------------------------------
# Knowledge base
# ---------------------------------------------------------

with knowledge_tab:
    st.subheader("Institute knowledge base")

    st.write("Documents directory:")
    st.code(str(documents_dir))

    if document_files:
        for document in document_files:
            st.write(f"📄 {document.name}")
    else:
        st.info("No supported institute documents found.")

    st.metric("Indexed chunks", len(rag_service.document_chunks))

    st.caption(
        "Document upload and re-indexing controls will be added next."
    )


# ---------------------------------------------------------
# Counselor follow-up queue
# ---------------------------------------------------------

with followup_tab:
    st.subheader("Counselor review queue")

    st.caption(
        "The queue is session-only and is not persisted to a database yet."
    )

    if not st.session_state.followups:
        st.info("No inquiries have been flagged for follow-up.")
    else:
        if st.button("Clear session queue"):
            st.session_state.followups = []
            st.rerun()

        for index, item in enumerate(st.session_state.followups, start=1):
            with st.expander(f"Inquiry {index}: {item['question'][:80]}"):
                st.markdown("**Student question**")
                st.write(item["question"])

                st.markdown("**AI answer**")
                st.write(item["answer"])

                st.write("Answer mode:", item["answer_mode"])

                st.markdown("**Sources**")
                if item["sources"]:
                    for source in item["sources"]:
                        st.write(
                            f"{source.get('source', 'Unknown')} — "
                            f"chunk {source.get('chunk_id', 'N/A')}"
                        )
                else:
                    st.write("No source documents recorded.")
"""

# Validate the complete app before overwriting app.py.
app_path = PROJECT_DIR / "app.py"
compile(APP_SOURCE, str(app_path), "exec")

app_path.write_text(APP_SOURCE, encoding="utf-8")

print("Created:", app_path)
print("Python syntax check passed.")
print("No API requests were made.")

Created: C:\Hehe\AI-Lead-Desk-Runtime-Errors\app.py
Python syntax check passed.
No API requests were made.
